In [109]:
import pandas as pd
df = pd.read_excel(r'D:\PYTHON\Machine Learning\Data_set\Heart_Disease_Custom_Dataset_600_Rows.xlsx')
df = df.drop_duplicates()
df.columns = df.columns.str.strip().str.lower()

In [110]:
df['chest_pain'] = df['chest_pain'].str.strip().str.lower()
df['sex'] = df['sex'].str.strip().str.lower()
df['exercise_angina'] = df['exercise_angina'].str.strip().str.lower()
df['smoking'] = df['smoking'].str.strip().str.lower()
df['family_history'] = df['family_history'].str.strip().str.lower()

In [111]:
import numpy as np
numeric_cols = df.select_dtypes(include='number').columns
df[numeric_cols] = df[numeric_cols].mask(df[numeric_cols] < 0, np.nan)

In [112]:
from sklearn.model_selection import train_test_split

unnecessary_col = ['patient_id','heart_disease']

X = df.drop(unnecessary_col, axis=1)
y = df['heart_disease']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [113]:
print(df.isnull().sum().sum())

0


In [114]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

categorical_cols = X.select_dtypes(include='object').columns

for col in categorical_cols:
    X_train[col] = le.fit_transform(X_train[col])
    X_test[col] = le.transform(X_test[col])

C:\Users\ruida\AppData\Local\Temp\ipykernel_20324\572337083.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include='object').columns


In [115]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

X_train_encoded = encoder.fit_transform(X_train[['chest_pain']])
X_test_encoded = encoder.transform(X_test[['chest_pain']])

In [116]:
X_train_encoded = pd.DataFrame(
    X_train_encoded,
    columns=encoder.get_feature_names_out(['chest_pain']),
    index=X_train.index
)

X_test_encoded = pd.DataFrame(
    X_test_encoded,
    columns=encoder.get_feature_names_out(['chest_pain']),
    index=X_test.index
)

In [117]:
X_train = X_train.drop('chest_pain', axis=1)
X_test = X_test.drop('chest_pain', axis=1)

X_train = pd.concat([X_train, X_train_encoded], axis=1)
X_test = pd.concat([X_test, X_test_encoded], axis=1)

In [118]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [119]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=5)

model.fit(X_train_scaled, y_train)
y_pred_knn = model.predict(X_test_scaled)
print(le.inverse_transform(y_pred_knn)[0])

yes


In [120]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

print("Accuracy :", accuracy_score(y_test, y_pred_knn))
print("Precision :", precision_score(y_test, y_pred_knn))
print("Recall :", recall_score(y_test, y_pred_knn))
print("Confusion Matrix :")
print(confusion_matrix(y_test, y_pred_knn))

Accuracy : 0.8416666666666667
Precision : 0.9230769230769231
Recall : 0.875
Confusion Matrix :
[[17  7]
 [12 84]]


In [121]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(X_train_scaled, y_train)
y_pred_lor = model.predict(X_test_scaled)

print(le.inverse_transform(y_pred_lor)[0])

no


In [122]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

print("Accuracy :", accuracy_score(y_test, y_pred_lor))
print("Precision :", precision_score(y_test, y_pred_lor))
print("Recall :", recall_score(y_test, y_pred_lor))
print("Confusion Matrix :")
print(confusion_matrix(y_test, y_pred_lor))

Accuracy : 0.9666666666666667
Precision : 0.9791666666666666
Recall : 0.9791666666666666
Confusion Matrix :
[[22  2]
 [ 2 94]]


In [123]:
print("Conclusion:")
print("Among the two models, Logistic Regression achieved the highest accuracy.")
print("Therefore, Logistic Regression is selected as the final model for heart disease prediction.")

Conclusion:
Among the two models, Logistic Regression achieved the highest accuracy.
Therefore, Logistic Regression is selected as the final model for heart disease prediction.


In [124]:
import joblib

joblib.dump(model, "model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(encoder, "encoder.pkl")

print("Saved Successfully!")

Saved Successfully!
